# 01c. Practice Set 1

**No new material here:** only practice on notebooks 01a and 01b. The exercises go roughly from
easy to hard:

- **Part A**, by hand: probabilities, log-loss, a gradient-descent step.
- **Part B**, reading a model: boundaries, sides, and a matching game.
- **Part C**, a small project from start to finish, including choosing a threshold by **cost**.
- **Part D**, evaluation: confusion matrix, measures, imbalanced data, AUC.
- **Part E**, many features: reading weights and odds.
- **Challenge**, for those who want more.

For the "by hand" exercises, really do them on paper first (a calculator for $e^x$ and $\log$ is
fine), and only then type the numbers in. The point is to practice the thinking, not the typing.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Interactive sliders (optional): if ipywidgets is missing, we show a few fixed pictures instead.
try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets is not installed: the interactive cells will show fixed pictures instead.")

from sklearn.linear_model import LogisticRegression

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# --- exercise checker: just run this cell -----------------------------------------------
import base64 as _b64, json as _json
_ANS = _json.loads(_b64.b64decode("eyJBMSI6IFstMC41LCAxLjAsIC0wLjUsIDAuMzc3NTQwNjY4Nzk4MTQ1NCwgMC43MzEwNTg1Nzg2MzAwMDQ5LCAwLjM3NzU0MDY2ODc5ODE0NTRdLCAiQTIiOiAwLjkyMDQ3MTg4NTI5MjgxMiwgIkEzIjogWzAuMjc5ODg1OTQyMDE5Mzg1MDUsIC0wLjI4OTEyNTk5Nzg2ODUyMTMsIDAuMTYyMDQ2NjM4NzQyMDk4NTddLCAiQTQiOiBbMC4zNjAwNTcwMjg5OTAzMDc0NSwgLTAuMzU1NDM3MDAxMDY1NzM5NCwgLTAuMDgxMDIzMzE5MzcxMDQ5MjgsIDAuODMxMjE0MzM4MDgzMzkyNV0sICJCMSI6IFsxLCAwLCAxLCAxXSwgIkIyIjogWzAuNSwgLTEuMF0sICJCMyI6IFswLjczMTA1ODU3ODYzMDAwNDksIDEuNTk4NjEyMjg4NjY4MTA5OF0sICJCNCI6IFsiYyIsICJhIiwgImIiLCAiZCJdLCAiQzMiOiAwLjcyLCAiQzNiIjogWzExNCwgMzksIDQ1LCAxMDJdLCAiQzQiOiAwLjc1LCAiRDEiOiBbMywgMiwgMiwgMywgMC42LCAwLjYsIDAuNiwgMC42XSwgIkQyIjogWzAuNiwgMC41NzE0Mjg1NzE0Mjg1NzE0LCAwLjgsIDAuNjY2NjY2NjY2NjY2NjY2Nl0sICJEMyI6IFswLjk5LCAwLjAsIDAuOTc2LCAwLjI2NjY2NjY2NjY2NjY2NjY2LCAwLjhdLCAiRDQiOiAwLjY0LCAiRTIiOiAwLjg1MTk1MjgwMTk2ODMxMDYsICJFMyI6IFszLjMyMDExNjkyMjczNjU0NzIsIDAuNDQ5MzI4OTY0MTE3MjIxNTZdLCAiQ0giOiAxLjA5ODYxMjI4ODY2ODEwOTh9"))

def _has_str(v):
    return isinstance(v, str) or (isinstance(v, (list, tuple)) and any(_has_str(x) for x in v))

def _norm(v):
    return [_norm(x) for x in v] if isinstance(v, (list, tuple, np.ndarray)) else str(v).strip().lower()

def _close(a, b, tol):
    if _has_str(b):
        return _norm(a) == _norm(b)
    try:
        return np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol, rtol=0)
    except Exception:
        return False

def check(key, value, tol=1e-3):
    # Compare your answer for exercise `key` with the expected one.
    if value is None or value is Ellipsis or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, _ANS[key], tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_against(key, value, reference, tol=1e-3):
    # Compare your answer with a reference value computed in the same cell.
    if value is None or (isinstance(value, (list, tuple)) and any(v is None for v in value)):
        print(f"⏳ {key}: not answered yet")
    elif _close(value, reference, tol):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. You got {value}; check your work and try again")

def check_fn(key, fn, tol=1e-3):
    # Call your function on some test inputs and compare with the expected outputs.
    spec = _ANS[key]
    try:
        outs = [fn(*[np.array(a, dtype=float) for a in args]) for args in spec["args"]]
    except Exception as e:
        print(f"❌ {key}: your function raised an error: {type(e).__name__}: {e}")
        return
    if any(o is None for o in outs):
        print(f"⏳ {key}: not answered yet (your function returns None)")
    elif all(_close(o, ex, tol) for o, ex in zip(outs, spec["out"])):
        print(f"✅ {key}: correct!")
    else:
        print(f"❌ {key}: not yet. Your function gives {outs}; check your work and try again")

## Part A: by hand

A tiny dataset with $N = 3$ examples and $n = 2$ features:

| example | $x_1$ | $x_2$ | $y$ |
|---|---|---|---|
| 1 | 1 | 2 | 1 |
| 2 | 2 | 0 | 0 |
| 3 | 0 | 1 | 0 |

The current model has $w = (0.5, -0.5)$ and $b = 0$.

In [ ]:
X_A = np.array([[1.0, 2.0], [2.0, 0.0], [0.0, 1.0]])
y_A = np.array([1, 0, 0])
w_A = np.array([0.5, -0.5])
b_A = 0.0

### A1. Scores and probabilities

Compute $z_i = w^T x_i + b$ and $h_i = \sigma(z_i)$ for the three examples. Store them in `z_A` and
`h_A` (arrays of length 3).

In [ ]:
# YOUR CODE HERE
z_A = None
h_A = None
print("z:", z_A, "\nh:", h_A)
check("A1", None if z_A is None or h_A is None else list(z_A) + list(h_A))

### A2. The log-loss

Compute the average log-loss of the model on the three examples. Store it in `loss_A`. Which example
contributes the most, and why?

In [ ]:
# YOUR CODE HERE
loss_A = None
print(loss_A)
check("A2", loss_A)

**Your answer:** *(write here)*

### A3. The gradient

Compute the gradient of the average log-loss, $\frac{\partial L}{\partial w_1}$,
$\frac{\partial L}{\partial w_2}$ and $\frac{\partial L}{\partial b}$, using the formula from 01a.
Store them in `grad_w_A` (an array of length 2) and `grad_b_A`.

In [ ]:
# YOUR CODE HERE
grad_w_A = None
grad_b_A = None
print(grad_w_A, grad_b_A)
check("A3", None if grad_w_A is None else list(grad_w_A) + [grad_b_A])

### A4. One step of gradient descent

Take **one** gradient-descent step with learning rate $\eta = 0.5$. Store the new parameters in
`w_A_new` and `b_A_new`, and the new log-loss in `loss_A_new`. Did the loss go down?

In [ ]:
# YOUR CODE HERE
w_A_new = None
b_A_new = None
loss_A_new = None
print("new w:", w_A_new, " new b:", b_A_new)
print("loss before:", loss_A, "  loss after:", loss_A_new)
check("A4", None if w_A_new is None else list(w_A_new) + [b_A_new, loss_A_new])

**Your answer:** *(write here)*

## Part B: reading a model

A model has $w = (2, -1)$ and $b = -1$. Use threshold 0.5.

### B1. Which side?

Without a computer, decide which of these points are predicted as **class 1**:
$P_1 = (1, 0)$, $P_2 = (0, 1)$, $P_3 = (2, 2)$, $P_4 = (0.5, 0)$.
Store the predictions (0 or 1) in `pred_B`, in that order.

*Careful with $P_4$!*

In [ ]:
# YOUR CODE HERE
pred_B = None
check("B1", pred_B)

### B2. Where does the boundary cross the axes?

The boundary is $2x_1 - x_2 - 1 = 0$. Where does it cross the $x_1$-axis (where $x_2 = 0$), and where
does it cross the $x_2$-axis (where $x_1 = 0$)? Store the two numbers in `cross_x1` and `cross_x2`.
Then draw the line and the four points: fill in the plotting cell.

In [ ]:
# YOUR CODE HERE
cross_x1 = None
cross_x2 = None
check("B2", [cross_x1, cross_x2])

In [ ]:
P = np.array([[1, 0], [0, 1], [2, 2], [0.5, 0]])
x1s = np.linspace(-1, 3, 100)
# YOUR CODE HERE
plt.scatter(P[:, 0], P[:, 1], s=80, c="gold", edgecolor="k", zorder=3)
for k, (px, py) in enumerate(P):
    plt.annotate(f"P{k + 1}", (px, py), textcoords="offset points", xytext=(8, 5))
plt.xlim(-1, 3); plt.ylim(-2, 3)
plt.xlabel("x1"); plt.ylabel("x2")
plt.title("Which side is class 1?")
plt.show()

**Your answer:** *(write here)*

### B3. Probabilities

1. What probability does the model give to $P_3 = (2, 2)$? Store it in `p_P3`.
2. Walking along the $x_1$-axis ($x_2 = 0$), at what $x_1$ does the probability reach **0.9**? Store it
   in `x1_09`.

In [ ]:
# YOUR CODE HERE
p_P3 = None
x1_09 = None
check("B3", [p_P3, x1_09])

### B4. The matching game

Below are four probability maps, **1 to 4**, drawn from four models, **a to d** (in a shuffled order):

| model | $w$ | $b$ |
|---|---|---|
| a | $(1, 0)$ | $0$ |
| b | $(0, -3)$ | $0$ |
| c | $(1, 1)$ | $-2$ |
| d | $(4, 4)$ | $-8$ |

Match each map to its model **without** computing anything; reason about the boundary's direction and
position, which side is red, and how sharp the transition is. Store your answer as a list of four
letters, `match_B` = [model of map 1, model of map 2, model of map 3, model of map 4].

In [ ]:
models = {"a": ((1, 0), 0), "b": ((0, -3), 0), "c": ((1, 1), -2), "d": ((4, 4), -8)}
order = list(np.array(list("abcd"))[np.random.default_rng(0).permutation(4)])   # a secret shuffle
g1, g2 = np.meshgrid(np.linspace(-3, 4, 200), np.linspace(-3, 4, 200))
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for k, (ax, name) in enumerate(zip(axes, order)):
    (w1, w2), bb = models[name]
    ax.contourf(g1, g2, sigmoid(w1 * g1 + w2 * g2 + bb), levels=np.linspace(0, 1, 21), cmap="coolwarm")
    ax.set_title(f"map {k + 1}")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
plt.show()
del order                                        # no peeking :)

In [ ]:
# YOUR CODE HERE
match_B = None
check("B4", match_B)

**Your answer:** *(write here)*

## Part C: a small project, from start to finish

A college wants to predict whether an applicant will **pass the first year** ($y = 1$) from two
entrance exams, **math** and **English** (each scored 0 to 100). We have data on 300 past students
(training) and on another 300 students (test).

In [ ]:
def make_applicants(n, seed):
    # Entrance-exam scores (math, English); y = 1 means the student passed the first year.
    rng = np.random.default_rng(seed)
    math = np.clip(rng.normal(70, 12, n), 30, 100).round()
    english = np.clip(rng.normal(70, 12, n), 30, 100).round()
    z = 0.12 * (math - 72) + 0.08 * (english - 68)                # (hidden from the learner!)
    y = (rng.uniform(size=n) < 1 / (1 + np.exp(-z))).astype(int)
    return np.column_stack([math, english]), y

X_app, y_app = make_applicants(300, seed=10)          # past students: training data
X_app_test, y_app_test = make_applicants(300, seed=11)  # a different group: test data

print(X_app[:5], y_app[:5])
print("fraction who passed (training):", y_app.mean().round(3))

### C1. Look at the data, and fit a model

1. Plot the training data (math on the x-axis, English on the y-axis, coloured by $y$).
2. Fit a `LogisticRegression` on the training data, and print $w$ and $b$. Which exam matters more?

Write all the code yourself. You can copy pieces from 01b.

In [ ]:
# YOUR CODE HERE
model = None

**Your answer:** *(write here)*

### C2. The probability map

Draw the probability map of your model over the training data, with the 0.5 boundary as a thick line.
(The data is noisy, so expect a lot of overlap.)

In [ ]:
# YOUR CODE HERE

### C3. Evaluate on the test data

For the **test** students, compute the accuracy (store it in `acc_app`) and the confusion counts
`[TP, FP, FN, TN]` at threshold 0.5 (store them in `cm_app`). You may use sklearn.

In [ ]:
# YOUR CODE HERE
acc_app = None
cm_app = None
print("accuracy:", acc_app, "  [TP, FP, FN, TN] =", cm_app)
check("C3", acc_app, tol=0.02)
check("C3b", cm_app, tol=3)

### C4. Choosing a threshold by cost

The college decides: admit an applicant if $h(x) \ge t$. Two kinds of mistake:

- **admitting** a student who then **fails** (a false positive) costs the college **3** units (wasted
  resources, and a hard year for the student);
- **rejecting** a student who **would have passed** (a false negative) costs **1** unit.

For each threshold in `thresholds`, compute the **total cost** on the **training** data. Store the
costs in `costs`, and the cheapest threshold in `best_t`. Plot the cost against the threshold.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
# YOUR CODE HERE
costs = None
best_t = None
print("best threshold:", best_t)
check("C4", best_t)

**Think about it:** the best threshold is **not** 0.5. Why is it higher? Can you guess a formula for
the best threshold in terms of the two costs? *(Hint: when admitting someone with probability $p$ of
passing, the expected cost of admitting is $3(1-p)$, and the expected cost of rejecting is
$1 \cdot p$. When is admitting cheaper?)*

**Your answer:** *(write here)*

## Part D: evaluation

A model gave these probabilities to 10 test examples:

| example | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| $y$ | 1 | 1 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 1 |
| $h$ | 0.95 | 0.8 | 0.7 | 0.6 | 0.55 | 0.4 | 0.35 | 0.2 | 0.1 | 0.05 |

In [ ]:
y_D = np.array([1, 1, 0, 1, 0, 0, 1, 0, 0, 1])
h_D = np.array([0.95, 0.8, 0.7, 0.6, 0.55, 0.4, 0.35, 0.2, 0.1, 0.05])

### D1. By hand, at threshold 0.5

Compute TP, FP, FN, TN, then accuracy, precision, recall and F1. Store them all in one list:
`D1 = [TP, FP, FN, TN, accuracy, precision, recall, F1]`. Do it **by hand** first.

In [ ]:
# YOUR CODE HERE
D1 = None
print(D1)
check("D1", D1)

### D2. At threshold 0.3

Recompute accuracy, precision, recall and F1 at threshold **0.3**:
`D2 = [accuracy, precision, recall, F1]`. What changed, and why?

In [ ]:
# YOUR CODE HERE
D2 = None
print(D2)
check("D2", D2)

**Your answer:** *(write here)*

### D3. Fraud detection: imbalanced data

A bank checks 1000 transactions, of which **10 are fraud** (the positive class).

- **Model A** flags no transactions at all.
- **Model B** flags 30 transactions, 8 of which are really fraud.

Compute the accuracy and recall of model A, and the accuracy, precision and recall of model B:
`D3 = [acc_A, recall_A, acc_B, precision_B, recall_B]`. Which model would the bank want?

In [ ]:
# YOUR CODE HERE
D3 = None
print(D3)
check("D3", D3)

**Your answer:** *(write here)*

### D4. AUC by counting pairs

Using the 10-example table from D1, compute the AUC by counting (positive, negative) pairs. There are
$5 \times 5 = 25$ pairs. Store the result in `auc_D`, then check it with `roc_auc_score`.

In [ ]:
# YOUR CODE HERE
auc_D = None
from sklearn.metrics import roc_auc_score
print("yours:", auc_D, "   sklearn:", roc_auc_score(y_D, h_D))
check("D4", auc_D)

## Part E: many features

An online shop trained a model to predict whether a visitor will **buy** something ($y = 1$). All four
features were put on the **same scale** (each has mean 0 and standard deviation 1) before training. The
learned weights:

| feature | meaning | weight |
|---|---|---|
| $x_1$ | number of visits this month | $1.2$ |
| $x_2$ | price of the item viewed | $-0.8$ |
| $x_3$ | day of the week | $0.05$ |
| $x_4$ | the item is already in the cart | $2.0$ |

and $b = -1$.

### E1. Reading the weights (in words)

1. Which feature has the strongest effect on the prediction?
2. Which feature pushes the prediction toward "won't buy"? Does that make sense?
3. Which feature barely matters? Would you keep it?

**Your answer:** *(write here)*

### E2. A prediction

A visitor has (on the standardized scale) $x = (1, 0.5, -1, 1)$. What is the probability they buy?
Store it in `p_E`.

In [ ]:
w_E = np.array([1.2, -0.8, 0.05, 2.0])
b_E = -1.0
x_E = np.array([1.0, 0.5, -1.0, 1.0])
# YOUR CODE HERE
p_E = None
print(p_E)
check("E2", p_E)

### E3. From weights to odds

In 01a we saw that increasing $x_j$ by 1 **adds** $w_j$ to the log-odds. What does it do to the
**odds** $\frac{p}{1-p}$ themselves?

1. By what factor are the odds multiplied when $x_1$ (visits) increases by 1? Store it in
   `odds_factor_visits`.
2. And when $x_2$ (price) increases by 1? Store it in `odds_factor_price`.

*Hint:* if $\log(\text{odds})$ grows by $w_j$, then the odds are multiplied by $e^{w_j}$.

In [ ]:
# YOUR CODE HERE
odds_factor_visits = None
odds_factor_price = None
print(odds_factor_visits, odds_factor_price)
check("E3", [odds_factor_visits, odds_factor_price])

**Your answer:** *(write here)*

## Challenge ★

We want a model whose boundary is the line $x_1 + x_2 = 2$, with class 1 above it, and which is **at
least 90% sure** about the point $(2, 2)$. Models of the form $w = c\,(1, 1)$, $b = -2c$ (with $c > 0$)
all have this boundary.

What is the **smallest** $c$ that works? Store it in `c_min`, and verify it by computing
$h(2, 2)$ for that $c$.

In [ ]:
# YOUR CODE HERE
c_min = None
check("CH", c_min)

## References and further reading

- Google Machine Learning Crash Course,
  [Logistic regression: test your knowledge](https://developers.google.com/machine-learning/crash-course/logistic-regression/test-your-knowledge):
  more short practice questions.
- For review: MLU-Explain's [Logistic Regression](https://mlu-explain.github.io/logistic-regression/)
  and [ROC & AUC](https://mlu-explain.github.io/roc-auc/), and Google's
  [Thresholds and the confusion matrix](https://developers.google.com/machine-learning/crash-course/classification/thresholding).